**1. Open the report** – Start Chrome with Selenium and open the SITUAS report.

In [ ]:
import pandas as pd
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

driver = webdriver.Chrome()     
driver.get("https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2024-12-31")

**2. Scraping the SITUAS table** – Read the 790 pages (10 municipalities each) and collect them in a table.

**Attention:** check that the rows counted equal the unique municipality codes (expected: 7,896 = 7,896). If the unique codes are fewer, some pages were read twice and others skipped: scrape again.

In [5]:
all_rows = []
previous_first = None                                 

for page in range(790):
    page_rows = []
    while not page_rows or page_rows[0][4] == previous_first:    
        page_rows = []
        soup = BeautifulSoup(driver.page_source, "html.parser")
        for tr in soup.find_all("tr"):
            cells = [td.get_text(strip=True) for td in tr.find_all("td")]
            if len(cells) == 17 and cells[0] != "":
                page_rows.append(cells)

    all_rows.extend(page_rows)                           
    previous_first = page_rows[0][4]                     

    button = driver.find_element(By.CLASS_NAME, "ant-pagination-next")
    driver.execute_script("arguments[0].click();", button)

df_scraped = pd.DataFrame(all_rows)
print("Rows collected:", len(df_scraped), "| Different codes:", df_scraped[4].nunique())  
print(df_scraped[[4, 6]].head(1))
print(df_scraped[[4, 6]].tail(1))

Rows collected: 7896 | Different codes: 7896
        4      6
0  001001  Agliè
           4              6
7895  111107  Villaspeciosa


**3. Save** – Give the 17 column names and save the table. 

In [6]:
df_scraped.columns = [
    "Codice Ripartizione geografica", "Codice Regione", "Codice Provincia (Storico)",
    "Codice Provincia/Uts", "Codice Comune (alfanumerico)", "Codice Comune (numerico)",
    "Comune", "Comune (dizione straniera)", "Sigla automobilistica",
    "Capoluogo di Provincia/Uts", "Capoluogo di Regione", "Popolazione legale",
    "Anno Censimento", "Superficie (Kmq)", "Anno (Superficie)",
    "Popolazione residente", "Anno (Popolazione residente)",]
df_scraped.to_csv("Data/Raw/situas_scraped_2024-12-31.csv", sep=";", index=False, encoding="utf-8-sig")
print(df_scraped.shape)      
df_scraped.head(3)

(7896, 17)


,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
0,1,01,001,201,001001,1001,Agliè,,TO,0,0,2562,2021,"13,1463",2024,2585,2024
1,1,01,001,201,001002,1002,Airasca,,TO,0,0,3660,2021,"15,7393",2024,3695,2024
2,1,01,001,201,001003,1003,Ala di Stura,,TO,0,0,467,2021,"46,3316",2024,463,2024


**4. Close the browser**

In [7]:
driver.quit()

**5. Compare with the official export (default reading)** – Both files read with the default settings: same shape (7,896 × 17) but `Identical: False`. Why?

In [23]:
df_before = pd.read_csv("Data/Raw/situas_scraped_2024-12-31.csv", sep=";")
df_export_before = pd.read_csv("Data/Raw/situas_municipalities_2024-12-31.csv", sep=";")
print("Before fix shape:", df_before.shape)
print("Export before fix shape:", df_export_before.shape)
print("Identical:", df_before.equals(df_export_before))

Before fix shape: (7896, 17)
Export before fix shape: (7896, 17)
Identical: False


**6. Where do they differ?**
– Comparing the two default readings cell by cell, only `Superficie (Kmq)` differs (781 rows, about 10%): the export always writes 4 decimals (`9,3090`) while the web page drops the trailing zero (`9,309`). All the other columns, population included, are identical.

In [ ]:
txt_export = pd.read_csv("Data/Raw/situas_municipalities_2024-12-31.csv", sep=";", dtype=str, keep_default_na=False)
txt_scraped = pd.read_csv("Data/Raw/situas_scraped_2024-12-31.csv", sep=";", dtype=str, keep_default_na=False)
print("Shapes:", txt_export.shape, txt_scraped.shape)                            
diff = df_before.compare(df_export_before)     
print(diff.shape)                              
print(diff.head(6))

Shapes: (7896, 17) (7896, 17)
(781, 2)
   Superficie (Kmq)          
               self     other
8             9,309    9,3090
20          132,203  132,2030
24           94,082   94,0820
28           13,419   13,4190
32             6,39    6,3900
34           11,135   11,1350


**7. `info()` of the default reading** – `Superficie (Kmq)` is text (`str`) because of the decimal comma. `Sigla automobilistica` (7,804) and `Comune` (7,895) have fewer values than the 7,896 rows: some values are read as missing.

In [25]:
df_before.info()

<class 'pandas.DataFrame'>
RangeIndex: 7896 entries, 0 to 7895
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   Codice Ripartizione geografica  7896 non-null   int64
 1   Codice Regione                  7896 non-null   int64
 2   Codice Provincia (Storico)      7896 non-null   int64
 3   Codice Provincia/Uts            7896 non-null   int64
 4   Codice Comune (alfanumerico)    7896 non-null   int64
 5   Codice Comune (numerico)        7896 non-null   int64
 6   Comune                          7895 non-null   str  
 7   Comune (dizione straniera)      124 non-null    str  
 8   Sigla automobilistica           7804 non-null   str  
 9   Capoluogo di Provincia/Uts      7896 non-null   int64
 10  Capoluogo di Regione            7896 non-null   int64
 11  Popolazione legale              7896 non-null   int64
 12  Anno Censimento                 7896 non-null   int64
 13  Superficie (Km

**8. Fix** – Read the file again with `decimal=","` (the area becomes a number) and with `keep_default_na=False` and `na_values=[""]` (only empty cells are missing). Now `Comune` and `Sigla automobilistica` have 7,896 values and `Superficie (Kmq)` is `float64`.

In [26]:
df_situas = pd.read_csv("Data/Raw/situas_scraped_2024-12-31.csv",
    sep=";",                   
    decimal=",",                
    keep_default_na=False,     
    na_values=[""],             
)
df_situas.info()

<class 'pandas.DataFrame'>
RangeIndex: 7896 entries, 0 to 7895
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Codice Ripartizione geografica  7896 non-null   int64  
 1   Codice Regione                  7896 non-null   int64  
 2   Codice Provincia (Storico)      7896 non-null   int64  
 3   Codice Provincia/Uts            7896 non-null   int64  
 4   Codice Comune (alfanumerico)    7896 non-null   int64  
 5   Codice Comune (numerico)        7896 non-null   int64  
 6   Comune                          7896 non-null   str    
 7   Comune (dizione straniera)      124 non-null    str    
 8   Sigla automobilistica           7896 non-null   str    
 9   Capoluogo di Provincia/Uts      7896 non-null   int64  
 10  Capoluogo di Regione            7896 non-null   int64  
 11  Popolazione legale              7896 non-null   int64  
 12  Anno Censimento                 7896 non-null

**9. Dirty data: what was wrong and how we found it** 
– `isna().sum()` shows 1 missing name and 92 missing sigla (`Comune (dizione straniera)` has 7,772 empty cells: expected, it is filled only for municipalities with an official name in another language). 
The 92 rows share the same province code (263); one row has a missing name (code 1168). After the fix the same rows show the text "NA" (sigla of Napoli) and the town called "None": the values were in the file, and by default pandas reads "NA" and "None" as missing values.

In [27]:
print(df_before.isna().sum())

Codice Ripartizione geografica       0
Codice Regione                       0
Codice Provincia (Storico)           0
Codice Provincia/Uts                 0
Codice Comune (alfanumerico)         0
Codice Comune (numerico)             0
Comune                               1
Comune (dizione straniera)        7772
Sigla automobilistica               92
Capoluogo di Provincia/Uts           0
Capoluogo di Regione                 0
Popolazione legale                   0
Anno Censimento                      0
Superficie (Kmq)                     0
Anno (Superficie)                    0
Popolazione residente                0
Anno (Popolazione residente)         0
dtype: int64


In [28]:
missing_sigla = df_before[df_before["Sigla automobilistica"].isna()]
print(missing_sigla[["Comune", "Codice Provincia/Uts", "Sigla automobilistica"]].head(3))                       
print(df_situas.loc[missing_sigla.index, ["Comune", "Codice Provincia/Uts", "Sigla automobilistica"]].head(3))   
missing_name = df_before[df_before["Comune"].isna()]
print(missing_name[["Codice Comune (numerico)", "Comune", "Sigla automobilistica"]])                             
print(df_situas.loc[missing_name.index, ["Codice Comune (numerico)", "Comune", "Sigla automobilistica"]])       

        Comune  Codice Provincia/Uts Sigla automobilistica
5056    Acerra                   263                   NaN
5057  Afragola                   263                   NaN
5058   Agerola                   263                   NaN
        Comune  Codice Provincia/Uts Sigla automobilistica
5056    Acerra                   263                    NA
5057  Afragola                   263                    NA
5058   Agerola                   263                    NA
     Codice Comune (numerico) Comune Sigla automobilistica
164                      1168    NaN                    TO
     Codice Comune (numerico) Comune Sigla automobilistica
164                      1168   None                    TO


**10. Ricompare with the official export** 

In [29]:
df_export = pd.read_csv("Data/Raw/situas_municipalities_2024-12-31.csv", sep=";", decimal=",", thousands=".", keep_default_na=False, na_values=[""])

print("Export shape:", df_export.shape)            
print("Scraped shape:", df_situas.shape)            
print("Identical:", df_export.equals(df_situas))

Export shape: (7896, 17)
Scraped shape: (7896, 17)
Identical: True


**11. Keep the useful columns and save** – Municipality code (same as `REF_AREA` in the accidents table), name, province, region, capoluogo, resident population and area, saved in `Data/Processed/situas_clean.csv`.

In [ ]:
df_situas = df_situas[[
    "Codice Comune (numerico)", "Comune", "Sigla automobilistica", "Codice Regione",
    "Capoluogo di Provincia/Uts", "Capoluogo di Regione",
    "Popolazione residente", "Superficie (Kmq)",
]]
os.makedirs("Data/Processed", exist_ok=True)      
df_situas.to_csv("Data/Processed/situas_clean.csv", index=False)
print(df_situas.shape)      
df_situas.head(3)

(7896, 8)


,Codice Comune (numerico),Comune,Sigla automobilistica,Codice Regione,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione residente,Superficie (Kmq)
0,1001,Agliè,TO,1,0,0,2585,13.1463
1,1002,Airasca,TO,1,0,0,3695,15.7393
2,1003,Ala di Stura,TO,1,0,0,463,46.3316
